In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

I0000 00:00:1783419379.818650    8704 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.


In [2]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    # 1. Calcolo FLASH (4 byte per Float32, 1 byte per INT8)
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    # 2. Calcolo SRAM (Tensor Arena) con logica Adiacente (Buffer Reuse)
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    # Memoria occupata dal layer precedente (inizializzata con la dimensione dell'input)
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            # Per i layer come "Concatenate" che potrebbero avere output multipli/strani
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        # IL FIX È QUI: Sommiamo il layer precedente e il layer corrente!
        # È il momento esatto in cui TFLM consuma più RAM durante l'esecuzione di questo layer.
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

I0000 00:00:1783419383.372988    8704 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2603 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [12]:
# ==============================================================================
# DATA ENGINE V9 (Caricamento Globale in RAM)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA viene calcolato qui, una volta per tutte, in perfetto ordine cronologico!
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    # Uniamo tutte le liste in due immensi tensori Numpy pronti per la GPU
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
#val_indices = [23, 20, 0, 13, 9] 
#train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato. (7500 frame pre-calcolati)
File 2/18 processato. (7500 frame pre-calcolati)
File 3/18 processato. (7500 frame pre-calcolati)
File 4/18 processato. (7500 frame pre-calcolati)
File 5/18 processato. (7500 frame pre-calcolati)
File 6/18 processato. (7500 frame pre-calcolati)
File 7/18 processato. (7500 frame pre-calcolati)
File 8/18 processato. (7500 frame pre-calcolati)
File 9/18 processato. (7500 frame pre-calcolati)
File 10/18 processato. (7500 frame pre-calcolati)
File 11/18 processato. (7500 frame pre-calcolati)
File 12/18 processato. (7500 frame pre-calcolati)
File 13/18 processato. (7500 frame pre-calcolati)
File 14/18 processato. (7500 frame pre-calcolati)
File 15/18 processato. (7500 frame pre-calcolati)
File 16/18 processato. (7500 frame pre-calcolati)
File 17/18 processato. (7500 frame pre-calcolati)
File 18/18 processato. (7500 frame pre-calcolati)

--- PREPARAZIONE VAL

In [ ]:
# ==============================================================================
# 1. DATA ENGINE V12 (Caricamento Globale in RAM, LOG1P per segnali deboli) 
# ==============================================================================

def load_and_process_all_files(file_list, alpha=0.002):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   # Shape: (T, 6, 3, 120, 2)
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        
        # TRUCCO MAGISTRALE: Shape (T, 6, 120, 3)
        mag_reshaped = np.transpose(mag, (0, 1, 3, 2)) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # EMA
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
            
        # ====================================================================
        # FIX FISICA RADAR: Compressione logaritmica (salva le persone a 6m)
        # ====================================================================
        decluttered = np.log1p(decluttered)
        
        # Flatten delle coordinate
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)

    return X, Y

# ==============================================================================
# 2. SPLIT E NORMALIZZAZIONE RIGOROSA
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

print("\n--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---")
train_mean = np.mean(X_train_raw)
train_std = np.std(X_train_raw)

print(f"!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!")
print(f"ATTENZIONE: Questi numeri sono cambiati per via del log1p!")
print(f"MEAN: {train_mean:.6f}")
print(f"STD:  {train_std:.6f}")

X_train = (X_train_raw - train_mean) / (train_std + 1e-7)
X_val = (X_val_raw - train_mean) / (train_std + 1e-7)

del X_train_raw
del X_val_raw


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---
!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!
ATTENZIONE: Questi numeri sono cambiati per via del log1p!
MEAN: 1.915614
STD:  1.087612


In [18]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - RESIDUAL REDUCTION MODULES (RRM)
# ==============================================================================

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d(x, filters, r=8, name_prefix=""):
    """
    RRM: Red(Res(x)) 
    Unisce una skip connection pesata dal SE block e un dimezzamento 
    parallelo della dimensione temporale (range bins).
    """
    # --- 1. Residual Branch (Res) ---
    res = layers.Conv2D(filters, kernel_size=(1, 3), padding='same', activation='relu', 
                        name=f"{name_prefix}_res_conv")(x)
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x]) # Skip connection

    # --- 2. Reduction Branch (Red) ---
    # Due convoluzioni parallele con stride=(1,2) per dimezzare i range bins (da 120->60->30...)
    red1 = layers.Conv2D(filters, kernel_size=(1, 3), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv1")(res)
    # Kernel 1x1 funge da projection mapping tipico delle ResNet
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 # Numero di filtri base: mantiene il modello piccolo e potente
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # 1. Feature Extraction Iniziale (allinea il numero di canali a F per far funzionare le Add)
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    # 2. Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    # Excitation: riduce e poi ri-espande per imparare
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    # 3. Testa della rete (Flatten + Dense)
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.25, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [ ]:
print(np.max(X_train), np.min(X_train), np.mean(X_train), np.std(X_train))

In [19]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 (RRM)
# ==============================================================================

# Inizializzazione del nuovo modello
model_rrm = build_eeai_model_v2_rrm()

# Compilazione 
model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("best_model_toscano2.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM ---")
history_rrm = model_rrm.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=32,
    shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~752.30 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO MODELLO RRM ---
Epoch 1/50


I0000 00:00:1783429446.388884   12187 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2141391__.51


4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - hungarian_mask_acc: 0.6775 - hungarian_rmse_metres: 2.0160 - loss: 42.7819

I0000 00:00:1783429467.273327   12188 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2141391__.51


4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - hungarian_mask_acc: 0.6777 - hungarian_rmse_metres: 2.0141 - loss: 42.7067

I0000 00:00:1783429470.109661   12188 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2163376__.20
I0000 00:00:1783429473.447053   12186 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2163376__.20



Epoch 1: val_loss improved from None to 1.17692, saving model to best_model_toscano2.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 31s 6ms/step - hungarian_mask_acc: 0.7482 - hungarian_rmse_metres: 1.1454 - loss: 7.5242 - val_hungarian_mask_acc: 0.8378 - val_hungarian_rmse_metres: 0.6531 - val_loss: 1.1769 - learning_rate: 0.0010
Epoch 2/50
4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - hungarian_mask_acc: 0.8734 - hungarian_rmse_metres: 0.6845 - loss: 1.1034
Epoch 2: val_loss improved from 1.17692 to 0.76448, saving model to best_model_toscano2.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 23s 5ms/step - hungarian_mask_acc: 0.8910 - hungarian_rmse_metres: 0.6423 - loss: 0.9783 - val_hungarian_mask_acc: 0.9281 - val_hungarian_rmse_metres: 0.5619 - val_loss: 0.7645 - learning_rate: 0.0010
Epoch 3/50
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - hungarian_mask_acc: 0.9290 - hungarian_rmse_metres: 0.5518 - loss: 0.7160
Epoch 3: val_loss improved from 0.76448 to 0.61339, saving model to best_model_toscano2.ke

 toscano val_hungarian_mask_acc: 0.9474 - val_hungarian_rmse_metres: 0.3487 CON SPLIT 2
 toscano1 alfa =0.20 val_hungarian_mask_acc: 0.9573 - val_hungarian_rmse_metres: 0.4597
 toscano2 
   alfa = 0.02 F=64 dropout layer = 0.25 val_hungarian_mask_acc: 0.9652 - val_hungarian_rmse_metres: 0.4053 !!!!!!! 
   alfa = 0.002 F=64 val_hungarian_mask_acc: 0.9650 - val_hungarian_rmse_metres: 0.4251
   alfa = 0.002 F=64 dropout layer = 0.35 val_hungarian_mask_acc: 0.9674 - val_hungarian_rmse_metres: 0.4467
   alfa = 0.02 F=64 dropout layer = 0.35 val_hungarian_mask_acc: 0.9599 - val_hungarian_rmse_metres: 0.4122
   alfa = 0.02 F=64 dropout layer = 0.30 val_hungarian_mask_acc: 0.9574 - val_hungarian_rmse_metres: 0.4200

 

In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.1 (RRM Version)
# ==============================================================================

file_target = "dataset/data/window_000011.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.002
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori del modello RRM dal file .keras ...")
    
    # Caricamento del modello aggiornato
    model_rrm_loaded = load_model(
        "best_model_toscano2.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_rrm_loaded.predict(decluttered, verbose=0)
    
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar V9 (RRM) | Frame: {frame_idx}/{T-1} | Window: 5", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V8)...
Caricamento dei pesi migliori del modello RRM dal file .keras ...


I0000 00:00:1783359328.862936   28823 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1279208__.5
I0000 00:00:1783359330.041415   28821 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1280017__.5


Dati pronti! Inizializzazione Radar...
